In [33]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [34]:
logboek = pd.read_csv("./logboek.csv", sep=";")
countries = logboek["land"].tolist()
leagues = logboek["competitienaam"].tolist()
seasons = logboek["jaar"].tolist()

In [35]:
index = 0
country = countries[index]
league = leagues[index]
season = seasons[index]
filename = f"{country}_{league}_{season}.csv"
df = pd.read_csv(f"./cleaned_data/{filename}")
df.head()

,round,date,team_home,team_away,full_text
0,38,25.05.2025,Bournemouth,Leicester,25.05.2025 | Bournemouth | Leicester | 2 | 0
1,38,25.05.2025,Fulham,Manchester City,25.05.2025 | Fulham | Manchester City | 0 | 2
2,38,25.05.2025,Ipswich,West Ham,25.05.2025 | Ipswich | West Ham | 1 | 3
3,38,25.05.2025,Liverpool,Crystal Palace,25.05.2025 | Liverpool | Crystal Palace | 1 | 1
4,38,25.05.2025,Manchester Utd,Aston Villa,25.05.2025 | Manchester Utd | Aston Villa | 2 | 0


In [36]:
def create_hapset(df):
    teams = list(set(df["team_home"].unique()) | set(df["team_away"].unique()))
    n = len(teams)
    team_to_index = {team: i for i, team in enumerate(teams)}
    r = df["round"].max()
    hapset = np.full((n, r), 2, dtype=int)

    for _, row in df.iterrows():
        home_team = row["team_home"]
        away_team = row["team_away"]
        round_num = row["round"] - 1
        hapset[team_to_index[home_team], round_num] = 1
        hapset[team_to_index[away_team], round_num] = 0
    
    if np.any(hapset == 2):
        print("Warning: Some matches are missing in the dataset.")
    
    return hapset, team_to_index

def get_opponent_schedule(df):
    """
        Constructs the opponent schedule as a 2D array the first column is the team itself and then the nex columns are the opponents in each round.
    """
    teams = list(set(df["team_home"].unique()) | set(df["team_away"].unique()))
    n = len(teams)
    r = df["round"].max()
    team_to_index = {team: i for i, team in enumerate(teams)}

    opp_sched = np.zeros((n, r + 1), dtype=object)

    # Fill the first column with team names
    for team, index in team_to_index.items():
        opp_sched[index, 0] = team
    
    # Fill the opponent schedule based on the matches in the DataFrame
    for _, row in df.iterrows():
        home_team = row["team_home"]
        away_team = row["team_away"]
        round_num = row["round"]
        opp_sched[team_to_index[home_team], round_num] = away_team
        opp_sched[team_to_index[away_team], round_num] = home_team

    # Check if any team has an empty opponent slot (indicating missing data)
    if np.any(opp_sched == 0):
        print("Warning: Some matches are missing in the dataset.")

    return opp_sched

In [37]:
hapset, team_to_index = create_hapset(df)

In [38]:
hapset.shape[1]

38

In [39]:
hap = hapset[0]
hap

array([1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1])

In [40]:
1-hap

array([0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

In [41]:
def count_breaks(hap):
    count = 0
    for i in range(1, len(hap)):
        if hap[i] == hap[i-1]:
            count += 1
    return count

def count_breaks_hapset(hapset):
    breaks_per_team = []
    for team_hap in hapset:
        breaks = count_breaks(team_hap)
        breaks_per_team.append(breaks)
    return breaks_per_team

def count_double_breaks(hap):
    count = 0
    for i in range(2, len(hap)):
        if hap[i] == hap[i-1] == hap[i-2]:
            count += 1
    return count



def is_complementary(hapset):
    n = hapset.shape[0]

    # Odd number of teams can never be complementary
    if n % 2 != 0:
        print("Odd number of teams, cannot be complementary.")
        return False
    
    # check for every hap if its complement exists
    # As all hapsets are feasible there are no duplicates, so we can just check if the complement exists for each hap
    complement_hapset = 1 - hapset
    for hap in hapset:
        if not any(np.array_equal(hap, complement_hap) for complement_hap in complement_hapset):
            return False
    
    return True

def calc_percentage_complementary(hapset):
    n = hapset.shape[0]
    compl_count = 0
    complement_hapset = 1 - hapset

    for hap in hapset:
        if any(np.array_equal(hap, complement_hap) for complement_hap in complement_hapset):
            compl_count += 1
    
    percentage_compl = (compl_count / 2) / n

    return percentage_compl

def calc_min_max_breaks_per_team(hapset):
    breaks_per_team = count_breaks_hapset(hapset)
    team_min = np.argmin(breaks_per_team)
    team_max = np.argmax(breaks_per_team)

    return (team_min, min(breaks_per_team)), (team_max, max(breaks_per_team))

def is_DRR(n, r):
    return r == 2*(n-1)

def calc_seperation(df):
    teams = list(set(df["team_home"].unique()) | set(df["team_away"].unique()))
    n = len(teams)
    r = df["round"].max()

    if not is_DRR(n, r):
        print("Not a DRR schedule, separation is not defined.")
        return None
    
    df = df.sort_values(by="round")
    min_dif = 1000

    for index, row in df.iterrows():
        h_team = row["team_home"]
        a_team = row["team_away"]
        curr_round = row["round"]

        # find the next match of the same teams
        next_match = df[((df["team_home"] == a_team) & (df["team_away"] == h_team)) & (df["round"] > curr_round)]

        if not next_match.empty:
            next_round = next_match["round"].values[0]
            dif = next_round - curr_round
            if dif < min_dif:
                min_dif = dif
    return min_dif


def is_phased(df):
    teams = list(set(df["team_home"].unique()) | set(df["team_away"].unique()))
    n = len(teams)
    r = df["round"].max()
    if not is_DRR(n, r):
        print("Not a DRR schedule, cannot be phased.")
        return False
    
    separation = calc_seperation(df)
    return separation >= n-1

def is_mirrored(df):
    if not is_phased(df):
        print("Not a phased schedule, cannot be mirrored.")
        return False
    
    matchups = np.sort(df[['team_home', 'team_away']].values, axis=1)
    df_copy = df.copy()
    df_copy["matchup"] = matchups[:, 0] + "_" + matchups[:, 1]

    gaps = df_copy.groupby("matchup")["round"].max() - df_copy.groupby("matchup")["round"].min()

    if any(gaps != (df["round"].max() - df["round"].min()) // 2):
        return False
    print("Schedule is mirrored.")
    return True

def is_english(df):
    if not is_phased(df):
        print("Not a phased schedule, cannot be English.")
        return False
    n = len(set(df["team_home"].unique()) | set(df["team_away"].unique()))

    for index, row in df.iterrows():
        h_team = row["team_home"]
        a_team = row["team_away"]
        curr_round = row["round"]

        if curr_round == n:
            break

        # find the next match of the same teams
        if curr_round == n - 1:
            next_match = df[((df["team_home"] == a_team) & (df["team_away"] == h_team)) & (df["round"] == curr_round + 1)]
        else:
            next_match = df[((df["team_home"] == a_team) & (df["team_away"] == h_team)) & (df["round"] == curr_round + n)]

        if next_match.empty:
            return False

    print("Schedule is English.")
    return True

def check_specific_match(df, team1, team2):
    return df[((df["team_home"] == team1) & (df["team_away"] == team2)) | ((df["team_home"] == team2) & (df["team_away"] == team1))]

def is_french(df):
    if not is_phased(df):
        print("Not a phased schedule, cannot be French.")
        return False
    n = len(set(df["team_home"].unique()) | set(df["team_away"].unique()))

    for index, row in df.iterrows():
        h_team = row["team_home"]
        a_team = row["team_away"]
        curr_round = row["round"]
        
        if curr_round == n:
            break

        # find the next match of the same teams
        if curr_round == 1:
            next_match = df[((df["team_home"] == a_team) & (df["team_away"] == h_team)) & (df["round"] == 2*n - 2)]
        else:
            next_match = df[((df["team_home"] == a_team) & (df["team_away"] == h_team)) & (df["round"] == curr_round + n - 2)]

        if next_match.empty:
            return False

    print("Schedule is French.")
    return True

def has_no_start_end_breaks(hapset):
    for hap in hapset:
        if hap[0] == hap[1] or hap[-1] == hap[-2]:
            return False
    print("Schedule has no start/end breaks.")
    return True

def has_no_double_breaks(hapset):
    for hap in hapset:
        if count_double_breaks(hap) > 0:
            return False
    print("Schedule has no double breaks.")
    return True

def calc_max_consec_H(hapset):
    max_consec_breaks = 0
    for hap in hapset:
        consec_breaks = 0
        for i in range(1, len(hap)):
            if hap[i] == hap[i-1]:
                consec_breaks += 1
                max_consec_breaks = max(max_consec_breaks, consec_breaks)
            else:
                consec_breaks = 0
    return max_consec_breaks

def check_AAHAA_occurence(hapset):
    r = hapset.shape[1]
    AAHAA = np.array([0,0,1,0,0])

    for hap in hapset:
        for i in range(r - 4):
            if np.array_equal(hap[i: i + 5], AAHAA):
                return True

    return False



In [42]:
# Clean schedule — no breaks for 2 teams, no double breaks, no start/end breaks, no AAHAA
hapset1 = np.array([
    [1,0,1,0,1,0,1],
    [0,1,0,1,0,1,0],
    [1,0,0,1,1,0,1],
    [0,1,1,0,0,1,0],
    [1,0,1,0,0,1,0],
    [0,1,0,1,1,0,1],
    [0,1,1,0,1,0,1],
    [1,0,0,1,0,1,0],
])

# Heavy breaks — double breaks, start/end breaks, no AAHAA
hapset2 = np.array([
    [1,1,1,0,0,1,0],
    [0,0,0,1,1,0,1],
    [1,0,1,0,1,0,1],
    [0,1,0,1,0,1,0],
    [1,1,0,1,0,0,1],
    [0,0,1,0,1,1,0],
    [1,0,0,0,1,1,1],
    [0,1,1,1,0,0,0],
])

# Contains AAHAA pattern (team 0, rounds 0-4), double breaks, start/end breaks
hapset3 = np.array([
    [0,0,1,0,0,1,0],
    [1,1,0,1,1,0,1],
    [1,0,1,0,1,0,1],
    [0,1,0,1,0,1,0],
    [1,0,0,1,0,0,1],
    [0,1,1,0,1,1,0],
    [0,1,1,1,0,0,1],
    [1,0,0,0,1,1,0],
])

# Moderate breaks — no double breaks, no start/end breaks, no AAHAA
hapset4 = np.array([
    [1,0,1,0,1,0,1],
    [0,1,0,1,0,1,0],
    [1,0,1,0,0,1,0],
    [0,1,0,1,1,0,1],
    [1,0,0,1,0,1,0],
    [0,1,1,0,1,0,1],
    [1,0,1,1,0,0,1],
    [0,1,0,0,1,1,0],
])

In [43]:
print(count_breaks_hapset(hapset4))
print(calc_min_max_breaks_per_team(hapset4))

[0, 0, 1, 1, 1, 1, 2, 2]
((np.int64(0), 0), (np.int64(6), 2))


In [44]:
matchups = np.sort(df[['team_home', 'team_away']].values, axis=1)
df_copy = df.copy()
df_copy["matchup"] = matchups[:, 0] + "_" + matchups[:, 1]

gaps = df_copy.groupby("matchup")["round"].max() - df_copy.groupby("matchup")["round"].min()

In [45]:
check_specific_match(df, "Southampton", "West Ham")

,round,date,team_home,team_away,full_text
59,33,19.04.2025,West Ham,Southampton,19.04.2025 | West Ham | Southampton | 1 | 1
209,18,26.12.2024,Southampton,West Ham,26.12.2024 | Southampton | West Ham | 0 | 1


In [46]:
gaps[gaps < 19]

matchup
Arsenal_Brentford         13
Arsenal_Brighton          17
Arsenal_Chelsea           18
Arsenal_Crystal Palace    17
Arsenal_Everton           15
                          ..
Southampton_Tottenham     15
Southampton_West Ham      15
Southampton_Wolves        18
Tottenham_Wolves          13
West Ham_Wolves           15
Name: round, Length: 110, dtype: int64

In [47]:
is_phased(df)

np.False_

In [48]:
### Canonical
def get_one_factors_from_opp_sched(opp_sched):
    """
        Creates the one-factor of every round from the opponent schedule.
        Returns this as a list (duplicate one factors possible)
    """
    n = opp_sched.shape[0]
    rounds = opp_sched.shape[1] - 1 # first col is team names
    one_factors = []
    

    for r in range(1, rounds + 1):
        round_set = set()
        for i in range(n):
            team = opp_sched[i, 0] # team name
            opp = opp_sched[i, r] # opponent of i in round r
            game = frozenset([team, opp])
            # if the game was already in the set it will not be added
            round_set.add(game) 
        one_factors.append(frozenset(round_set))

    return one_factors

def get_canonical_indexes(n, i):
    lookup = {}
    lookup[i] = n
    lookup[n] = i

    for k in range(1, n//2):
        team1 = (i + k) % (n-1)
        team2 = (i - k) % (n-1)

        team1 = n-1 if team1 == 0 else team1
        team2 = n-1 if team2 == 0 else team2

        lookup[team1] = team2
        lookup[team2] = team1

    return lookup

def convert_teamlist_to_onefactors(team_order):
    n = len(team_order)
    team_list = [0] + team_order
    one_factors = []

    for i in range(1, n):
        lookup = get_canonical_indexes(n, i)
        round_set = set()
        for key, value in lookup.items():
            round_set.add(frozenset([team_list[key], team_list[value]]))
        one_factors.append(frozenset(round_set))
    
    return one_factors
        


def build_canonical_one_factors(round1, round2, fixed_team, team_list):
    n = len(team_list)
    last_pos = n - 1
    team_order = [None] * n
    team_order[last_pos] = fixed_team
    opposites_r1 = {}
    opposites_r2 = {}
    F1 = get_canonical_indexes(n, 1)
    F2 = get_canonical_indexes(n, 2)

    # get team1 as opp of fixedteam in round 1
    for pair in round1:
        pair = list(pair)
        if fixed_team in pair:
            team_order[0] = pair[0] if pair[1] == fixed_team else pair[1]
        else: 
            # get opposites from round 1
            opposites_r1[pair[0]] = pair[1]
            opposites_r1[pair[1]] = pair[0]
    
    # get team2 from round 2
    for pair in round2:
        pair = list(pair)
        if fixed_team in pair:
            team_order[1] = pair[0] if pair[1] == fixed_team else pair[1]
        else: 
            # get opposites from round 2
            opposites_r2[pair[0]] = pair[1]
            opposites_r2[pair[1]] = pair[0]

    idx = 2
    while None in team_order:
        print(f"Current known order: {team_order}")
        # first fill in from round 1 by getting opposite of team2
        opp_idx = F1[idx]
        current_team = team_order[idx - 1]
        print(f"current_team: {current_team}")
        opp_team = opposites_r1[current_team]
        print(f"opp_team: {opp_team}")
        team_order[opp_idx - 1] = opp_team

        # then fill in from F
        idx = F2[opp_idx]
        print(f"new index = {idx}")
        new_team =  opposites_r2[opp_team]
        team_order[idx - 1] = new_team

    return convert_teamlist_to_onefactors(team_order)

def is_canonical_schedule(opp_sched):
    n = opp_sched.shape[0]
    r = opp_sched.shape[1] - 1
    list_of_teams = list(opp_sched[:, 0])

    one_factors_to_check = get_one_factors_from_opp_sched(opp_sched)

    if len(set(one_factors_to_check)) != n - 1:
        print("Too many different one factors in schedule")
        return False
    
    for r_ind1 in range(r):
        round1 = one_factors_to_check[r_ind1]
        for r_ind2 in range(r_ind1 + 1, r):
            round2 = one_factors_to_check[r_ind2]
            for fixed_team in list_of_teams:
                can_one_factors = build_canonical_one_factors(round1, round2, fixed_team, list_of_teams)
                if set(can_one_factors) == set(one_factors_to_check):
                    return True
    
    print("No matching one factors found!")
    return False
    


In [49]:
round1 = {frozenset(['H','A']), frozenset(['B','G']), frozenset(['C','F']), frozenset(['D','E'])}
round2 = {frozenset(['H','B']), frozenset(['C','A']), frozenset(['D','G']), frozenset(['E','F'])}
fixed_team = 'H'
team_list = ['A','B','C','D','E','F','G','H']

In [50]:
build_canonical_one_factors(round1, round2, fixed_team, team_list)

Current known order: ['A', 'B', None, None, None, None, None, 'H']
current_team: B
opp_team: G
new index = 4
Current known order: ['A', 'B', None, 'D', None, None, 'G', 'H']
current_team: D
opp_team: E
new index = 6
Current known order: ['A', 'B', None, 'D', 'E', 'F', 'G', 'H']
current_team: F
opp_team: C
new index = 1


[frozenset({frozenset({'D', 'E'}),
            frozenset({'A', 'H'}),
            frozenset({'C', 'F'}),
            frozenset({'B', 'G'})}),
 frozenset({frozenset({'E', 'F'}),
            frozenset({'D', 'G'}),
            frozenset({'B', 'H'}),
            frozenset({'A', 'C'})}),
 frozenset({frozenset({'C', 'H'}),
            frozenset({'A', 'E'}),
            frozenset({'F', 'G'}),
            frozenset({'B', 'D'})}),
 frozenset({frozenset({'C', 'E'}),
            frozenset({'D', 'H'}),
            frozenset({'A', 'G'}),
            frozenset({'B', 'F'})}),
 frozenset({frozenset({'A', 'B'}),
            frozenset({'E', 'H'}),
            frozenset({'D', 'F'}),
            frozenset({'C', 'G'})}),
 frozenset({frozenset({'B', 'C'}),
            frozenset({'F', 'H'}),
            frozenset({'A', 'D'}),
            frozenset({'E', 'G'})}),
 frozenset({frozenset({'G', 'H'}),
            frozenset({'B', 'E'}),
            frozenset({'C', 'D'}),
            frozenset({'A', 'F'})})]

In [51]:
set1 = set([1,2,3,4])
list1 = [set1]
set2 = set([2,3,4,1])
set2 in list1

True

In [52]:
### Balancedness

def is_balanced(hapset):
    r = hapset.shape[1]
    odd = r % 2 == 1
    for hap in hapset:
        if odd:
            if np.sum(hap) not in [r//2, r//2 + 1]:
                return False
        else:
            if np.sum(hap) != r//2:
                return False
    return True

def calc_k_balanced(hapset):
    k = 0
    r = hapset.shape[1]

    for i in range(2, r + 1):
        hapset_k = hapset[:, :i]
        homes = np.sum(hapset_k, axis=1)
        aways = i - homes
        balance_r = np.abs(homes - aways)
        k = max(k, np.max(balance_r))
    return k


def calc_g_balanced(hapset):
    g = 0
    r = hapset.shape[1]

    for i in range(2, r+1):
        hapset_k = hapset[:, :i]
        homes = np.sum(hapset_k, axis=1)
        g = max(g, np.max(homes) - np.min(homes))
    return g

In [53]:
hapset = np.array([[1, 1, 1, 0], [0, 0, 0, 1], [1, 0, 0, 0], [0, 0, 1, 1]])
hapset[:, :2]


array([[1, 1],
       [0, 0],
       [1, 0],
       [0, 0]])

In [61]:
# carry-over effect

def calc_carry_over_effect(opp_sched, wrap_around = True, normalized = True):
    n = opp_sched.shape[0]
    # only keep n-1 first rounds
    opp_sched = opp_sched[:,:n]
    r = opp_sched.shape[1] - 1

    carry_over_matrix = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            carry_count = 0
            
            # count carry-over
            if wrap_around: max_range = r+1 
            else: max_range = r
            for rnd in range(1, max_range):
                if rnd == r: # WRAP AROUND
                    opponent_i = opp_sched[i, rnd]
                    opponent_j = opp_sched[j, 1]
                else:
                    opponent_i = opp_sched[i, rnd]
                    opponent_j = opp_sched[j, rnd + 1]

                if opponent_i == opponent_j:
                    carry_count += 1

            carry_over_matrix[i,j] = carry_count

    print(carry_over_matrix)
    tot_carry_over = np.sum(np.square(carry_over_matrix))
    # n^3 − 7n2 + 18n − 12
    max_carry_possible = n ** 3 - 7 * n ** 2 + 18 * n - 12
    print(max_carry_possible)
    if normalized:
        return tot_carry_over/max_carry_possible, carry_over_matrix
    return tot_carry_over, carry_over_matrix

In [55]:
arr = np.array(range(5))
print(arr)
arr[1:4]

[0 1 2 3 4]


array([1, 2, 3])

In [62]:


# Het schema uit Tabel 6(a) van Goossens & Spieksma
opp_sched = np.array([
    ['A', 'C', 'F', 'B', 'D', 'E'],
    ['B', 'E', 'D', 'A', 'C', 'F'],
    ['C', 'A', 'E', 'F', 'B', 'D'],
    ['D', 'F', 'B', 'E', 'A', 'C'],
    ['E', 'B', 'C', 'D', 'F', 'A'],
    ['F', 'D', 'A', 'C', 'E', 'B']
], dtype=object)

In [63]:
print(calc_carry_over_effect(opp_sched))
print(calc_carry_over_effect(opp_sched, wrap_around=True, normalized=False))

[[0 1 3 0 1 0]
 [0 0 1 3 1 0]
 [0 0 0 1 1 3]
 [3 0 0 0 1 1]
 [1 1 1 1 0 1]
 [1 3 0 0 1 0]]
60
(np.float64(1.0), array([[0, 1, 3, 0, 1, 0],
       [0, 0, 1, 3, 1, 0],
       [0, 0, 0, 1, 1, 3],
       [3, 0, 0, 0, 1, 1],
       [1, 1, 1, 1, 0, 1],
       [1, 3, 0, 0, 1, 0]]))
[[0 1 3 0 1 0]
 [0 0 1 3 1 0]
 [0 0 0 1 1 3]
 [3 0 0 0 1 1]
 [1 1 1 1 0 1]
 [1 3 0 0 1 0]]
60
(np.int64(60), array([[0, 1, 3, 0, 1, 0],
       [0, 0, 1, 3, 1, 0],
       [0, 0, 0, 1, 1, 3],
       [3, 0, 0, 0, 1, 1],
       [1, 1, 1, 1, 0, 1],
       [1, 3, 0, 0, 1, 0]]))


In [64]:
# flexibility
def lambers_IP_checker_SRR(hapset: np.ndarray, team1: int, team2: int, l: int):
    n_teams = hapset.shape[0]
    model = gp.Model("briskorn_condition")
    model.Params.OutputFlag = 0
    
    x = {}
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            for r in range(n_teams - 1):
                x[i,j,r] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{r}")

    for i in range(n_teams):
        for p in range(n_teams - 1):
            sum_last = gp.quicksum(x[j, i, p] for j in range(n_teams) if j < i)
            sum_first = gp.quicksum(x[i, j, p] for j in range(n_teams) if j > i)
            model.addConstr(sum_first + sum_last == 1, name=f"Constr_One_Game_{i}_{p}")

    for i in range(n_teams):
        for j in range(i+1, n_teams):
            model.addConstr(
                gp.quicksum(x[i, j, p] for p in range(n_teams - 1)) == 1, 
                name=f"Constr_Pair_{i}_{j}"
            )
    
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            for r in range(n_teams - 1):
                if hapset[i, r] == hapset[j, r]:
                    model.addConstr(x[i,j,r] == 0, name=f"Constr_HAP_{i}_{j}_{r}")

    if team1 > team2:
        team1, team2 = team2, team1
    model.addConstr(x[team1, team2, l] == 1, name=f"Constr_Force_{team1}_{team2}_{l}")
    
    model.optimize()

    is_feasible = model.status == GRB.OPTIMAL
    schedule = []
    if is_feasible:
        schedule = [(i, j, r) for (i, j, r), var in x.items() if var.X > 0.5]
        
    model.dispose()
    return int(is_feasible), schedule

def lambers_IP_checker_DRR(hapset: np.ndarray, home_team: int, away_team: int, l: int):
    """
    hapset: (n_teams, 4n-2) array, 1=Home, 0=Away
    home_team, away_team: ordered — home_team hosts away_team
    l: round to force this match into
    """
    n_teams = hapset.shape[0]
    n_rounds = hapset.shape[1]  # should be 4*(n_teams//2) - 2

    model = gp.Model("drr_spread")
    model.Params.OutputFlag = 0

    # x[i,j,r] = 1 means team i hosts team j in round r (ordered)
    x = {}
    for i in range(n_teams):
        for j in range(n_teams):
            if i == j:
                continue
            for r in range(n_rounds):
                x[i, j, r] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{r}")

    # Each ordered match (i,j) is played exactly once
    for i in range(n_teams):
        for j in range(n_teams):
            if i == j:
                continue
            model.addConstr(
                gp.quicksum(x[i, j, r] for r in range(n_rounds)) == 1,
                name=f"match_{i}_{j}"
            )

    # Each team plays exactly one match per round
    for i in range(n_teams):
        for r in range(n_rounds):
            model.addConstr(
                gp.quicksum(x[i, j, r] for j in range(n_teams) if j != i)   # i at home
                + gp.quicksum(x[j, i, r] for j in range(n_teams) if j != i) # i away
                == 1,
                name=f"one_game_{i}_{r}"
            )

    # HAP compatibility:
    # x[i,j,r]=1 requires hapset[i,r]=1 (i is home) and hapset[j,r]=0 (j is away)
    for i in range(n_teams):
        for j in range(n_teams):
            if i == j:
                continue
            for r in range(n_rounds):
                if hapset[i, r] != 1 or hapset[j, r] != 0:
                    model.addConstr(x[i, j, r] == 0)

    # Force the specific match into round l
    model.addConstr(x[home_team, away_team, l] == 1,
                    name=f"force_{home_team}_{away_team}_{l}")

    model.optimize()

    is_feasible = model.status == GRB.OPTIMAL
    schedule = []
    if is_feasible:
        schedule = [(i, j, r) for (i, j, r), var in x.items() if var.X > 0.5]

    model.dispose()
    return int(is_feasible), schedule


def spread_calculator(hapset: np.ndarray, is_drr: bool) -> int:
    n_teams = hapset.shape[0]
    r_rounds = hapset.shape[1]
    matches_seen = set()
    total_spread = 0

    pairs = (
        [(i, j) for i in range(n_teams) for j in range(n_teams) if i != j]
        if is_drr else
        [(i, j) for i in range(n_teams) for j in range(i+1, n_teams)]
    )

    for i, j in pairs:
        for r in range(r_rounds):
            if is_drr:
                skip_condition = hapset[i, r] == 0 or hapset[j, r] == 1 or (i, j, r) in matches_seen
            else:
                skip_condition = hapset[i, r] == hapset[j, r] or (i, j, r) in matches_seen
            if skip_condition:
                continue
            if is_drr:
                feas, sched = lambers_IP_checker_DRR(hapset, i, j, r)
            else:
                feas, sched = lambers_IP_checker_SRR(hapset, i, j, r)
            if feas:
                for match in sched:
                    if match not in matches_seen:
                        total_spread += 1
                        matches_seen.add(match)
    return total_spread



In [65]:
# TODO: fixed part
def lambers_IP_FP_SRR(hapset: np.ndarray, team1: int, team2: int):
    """
    Returns 1 if match {team1, team2} is in the fixed part (infeasible to place it
    in two different rounds across two schedules), 0 otherwise.
    """
    n_teams = hapset.shape[0]
    n_rounds = n_teams - 1
    W = 2

    if team1 > team2:
        team1, team2 = team2, team1

    model = gp.Model("FP_SRR")
    model.Params.OutputFlag = 0

    x = {}
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                for r in range(n_rounds):
                    x[w, i, j, r] = model.addVar(vtype=GRB.BINARY, name=f"x_{w}_{i}_{j}_{r}")

    # Each match played exactly once per schedule
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for r in range(n_rounds)) == 1,
                    name=f"match_{w}_{i}_{j}"
                )

    # Each team plays exactly once per round per schedule
    for w in range(W):
        for i in range(n_teams):
            for r in range(n_rounds):
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for j in range(i + 1, n_teams))
                    + gp.quicksum(x[w, j, i, r] for j in range(i))
                    == 1,
                    name=f"one_game_{w}_{i}_{r}"
                )

    # HAP compatibility
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                for r in range(n_rounds):
                    if hapset[i, r] == hapset[j, r]:
                        model.addConstr(x[w, i, j, r] == 0)

    # Target match {team1, team2} must be in different rounds across the two schedules
    for r in range(n_rounds):
        model.addConstr(
            gp.quicksum(x[w, team1, team2, r] for w in range(W)) <= 1,
            name=f"diff_round_{r}"
        )

    model.optimize()

    is_fixed = model.status != GRB.OPTIMAL
    model.dispose()
    return int(is_fixed)


def lambers_IP_FP_DRR(hapset: np.ndarray, home_team: int, away_team: int):
    """
    Returns 1 if ordered match (home_team, away_team) is in the fixed part, 0 otherwise.
    """
    n_teams = hapset.shape[0]
    n_rounds = hapset.shape[1]
    W = 2

    model = gp.Model("FP_DRR")
    model.Params.OutputFlag = 0

    x = {}
    for w in range(W):
        for i in range(n_teams):
            for j in range(n_teams):
                if i == j:
                    continue
                for r in range(n_rounds):
                    x[w, i, j, r] = model.addVar(vtype=GRB.BINARY, name=f"x_{w}_{i}_{j}_{r}")

    # Each ordered match (i,j) played exactly once per schedule
    for w in range(W):
        for i in range(n_teams):
            for j in range(n_teams):
                if i == j:
                    continue
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for r in range(n_rounds)) == 1,
                    name=f"match_{w}_{i}_{j}"
                )

    # Each team plays exactly once per round per schedule
    for w in range(W):
        for i in range(n_teams):
            for r in range(n_rounds):
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for j in range(n_teams) if j != i)
                    + gp.quicksum(x[w, j, i, r] for j in range(n_teams) if j != i)
                    == 1,
                    name=f"one_game_{w}_{i}_{r}"
                )

    # HAP compatibility
    for w in range(W):
        for i in range(n_teams):
            for j in range(n_teams):
                if i == j:
                    continue
                for r in range(n_rounds):
                    if hapset[i, r] != 1 or hapset[j, r] != 0:
                        model.addConstr(x[w, i, j, r] == 0)

    # target match (home_team, away_team) must be in different rounds
    for r in range(n_rounds):
        model.addConstr(
            gp.quicksum(x[w, home_team, away_team, r] for w in range(W)) <= 1,
            name=f"diff_round_{r}"
        )

    model.optimize()

    is_fixed = model.status != GRB.OPTIMAL
    model.dispose()
    return int(is_fixed)


def fixed_part_calculator(hapset, is_drr):
    n_teams = hapset.shape[0]
    fp = 0
    for i in range(n_teams):
        for j in range(i + 1, n_teams):
            if is_drr:
                fp += lambers_IP_FP_DRR(hapset, i, j)
            else: 
                fp += lambers_IP_FP_SRR(hapset, i, j)
    print(f"Fixed part: {fp}/{int(n_teams * (n_teams-1)/2)}")
    return fp


In [66]:
CPS8 = np.array([
    [1, 0, 1, 0, 1, 0, 0],  # PA(7): break at round 7 -> ends with A
    [0, 1, 0, 1, 0, 1, 1],  # PH(7): complement
    [1, 0, 1, 0, 0, 1, 0],  # PA(5): break at round 5
    [0, 1, 0, 1, 1, 0, 1],  # PH(5): complement
    [1, 0, 0, 1, 0, 1, 0],  # PA(3): break at round 3
    [0, 1, 1, 0, 1, 0, 1],  # PH(3): complement
    [0, 0, 1, 0, 1, 0, 1],  # PH(1): break at round 1 -> starts HH so break at 1
    [1, 1, 0, 1, 0, 1, 0],  # PA(1): complement
])

fixed_part_calculator(CPS8, is_drr=False)

Set parameter Username
Set parameter LicenseID to value 2745714
Academic license - for non-commercial use only - expires 2026-11-27
Fixed part: 4/28


4

In [67]:
print(spread_calculator(CPS8, is_drr=False))        # expected: 88
print(fixed_part_calculator(CPS8, is_drr=False))              # expected: 4

88
Fixed part: 4/28
4
